Big Picture: Here I am loading in a file of the various destinations 

In [21]:
import geopandas as gpd
import pandas as pd
from shapely import wkt

subwaystops = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/MTA_Subway_Stations_and_Complexes_20260717.csv")
bstops = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/MTA_Bus_Stops_20260717.csv")
broutes = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/MTA_Bus_Routes_20260717.csv")

In [22]:
neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights"
}

boundaries = {}
for key, name_filter in neighborhoods.items():
    neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights"
}

In [23]:
subwaystops = gpd.GeoDataFrame(
    subwaystops,
    geometry=gpd.points_from_xy(subwaystops["Longitude"], subwaystops["Latitude"]),
    crs="EPSG:4326"
)

WORKING_CRS = 2263
subwaystops = subwaystops.to_crs(WORKING_CRS)

In [24]:
import os
os.makedirs("outputs", exist_ok=True)

subwaystops.to_file("outputs/subway_stops.geojson", driver="GeoJSON")

Joining data for Bus Stops

In [25]:
bstops["geometry"] = bstops["Georeference"].apply(wkt.loads)
bstops = gpd.GeoDataFrame(bstops, geometry="geometry", crs="EPSG:4326")

WORKING_CRS = 2263
bstops = bstops.to_crs(WORKING_CRS)

In [26]:
bstops["Valid To"] = pd.to_datetime(bstops["Valid To"])

bstops = bstops.sort_values("Valid To", ascending=False)
bstops = bstops.drop_duplicates(subset="Stop ID", keep="first")

print(f"Rows after dedup: {len(bstops)}")

Rows after dedup: 17493


In [27]:
broutes_type = broutes[["Route ID", "Route Type"]].drop_duplicates()

route_priority = {"Express": 0, "SBS": 1, "Limited": 2, "Local": 3, "School": 4}
broutes_type["priority"] = broutes_type["Route Type"].map(route_priority)
broutes_type = broutes_type.sort_values("priority").drop_duplicates(subset="Route ID", keep="first")
broutes_type = broutes_type.drop(columns="priority")

In [28]:
bstops = bstops.merge(broutes_type, on="Route ID", how="left")

print(f"Total rows: {len(bstops)}")
print(f"Unique locations: {bstops.geometry.astype(str).nunique()}")

Total rows: 17493
Unique locations: 17444


In [29]:
import os
os.makedirs("outputs", exist_ok=True)

bstops.to_file("outputs/bus_stops.geojson", driver="GeoJSON")

Now its time to do the store locations

In [30]:
import geopandas as gpd

boundaries = {
    "bedstuy": gpd.read_file("Data/boundary_bedstuy.geojson"),
    "jacksonheights": gpd.read_file("Data/boundary_jacksonheights.geojson")
}

In [31]:
category_map = {
    # Grocery (7-9)
    ("shop", "greengrocer"): ("Grocery", 7, 9),
    ("shop", "health_food"): ("Grocery", 7, 9),
    ("shop", "supermarket"): ("Grocery", 7, 9),
    ("shop", "wholesale"): ("Grocery", 7, 9),

    # Stopby (5-8)
    ("shop", "convenience"): ("Stopby", 5, 8),
    ("shop", "coffee"): ("Stopby", 5, 8),
    ("amenity", "cafe"): ("Stopby", 5, 8),

    # Restaurant (3-7)
    ("amenity", "fast_food"): ("Restaurant", 3, 7),
    ("amenity", "restaurant"): ("Restaurant", 3, 7),
    ("amenity", "food_court"): ("Restaurant", 3, 7),

    # Kids (1-7)
    ("amenity", "school"): ("Kids", 1, 7),

    # Shopping (2-4) — Clothing, shoes, accessories
    ("shop", "clothes"): ("Shopping", 2, 4),
    ("shop", "shoes"): ("Shopping", 2, 4),
    ("shop", "fashion_accessories"): ("Shopping", 2, 4),
    ("shop", "baby_goods"): ("Shopping", 2, 4),
    ("shop", "bag"): ("Shopping", 2, 4),
    ("shop", "fabric"): ("Shopping", 2, 4),
    ("shop", "jewelry"): ("Shopping", 2, 4),
    ("shop", "leather"): ("Shopping", 2, 4),
    ("shop", "sewing"): ("Shopping", 2, 4),
    ("shop", "shoe_repair"): ("Shopping", 2, 4),
    ("shop", "tailor"): ("Shopping", 2, 4),
    ("shop", "watches"): ("Shopping", 2, 4),
    ("shop", "wool"): ("Shopping", 2, 4),

    # Shopping (2-4) — Health and beauty
    ("shop", "beauty"): ("Shopping", 2, 4),
    ("shop", "chemist"): ("Shopping", 2, 4),
    ("shop", "cosmetics"): ("Shopping", 2, 4),
    ("shop", "erotic"): ("Shopping", 2, 4),
    ("shop", "hairdresser"): ("Shopping", 2, 4),
    ("shop", "hairdresser_supply"): ("Shopping", 2, 4),
    ("shop", "hearing_aids"): ("Shopping", 2, 4),
    ("shop", "herbalist"): ("Shopping", 2, 4),
    ("shop", "massage"): ("Shopping", 2, 4),
    ("shop", "medical_supply"): ("Shopping", 2, 4),
    ("shop", "nutrition_supplements"): ("Shopping", 2, 4),
    ("shop", "optician"): ("Shopping", 2, 4),
    ("shop", "perfumery"): ("Shopping", 2, 4),
    ("shop", "piercing"): ("Shopping", 2, 4),
    ("shop", "tattoo"): ("Shopping", 2, 4),

    # Shopping (2-4) — Stationery, gifts, books, newspapers
    ("shop", "anime"): ("Shopping", 2, 4),
    ("shop", "books"): ("Shopping", 2, 4),
    ("shop", "gift"): ("Shopping", 2, 4),
}

In [ ]:
import osmnx as ox
import pandas as pd

osm_results = []
for name, boundary in boundaries.items():
    boundary_4326 = boundary.to_crs("EPSG:4326")
    polygon = boundary_4326.geometry.iloc[0]

    features = ox.features.features_from_polygon(polygon, tags={"amenity": True, "shop": True})
    features = features.reset_index()
    features = features.to_crs(WORKING_CRS)
    features["nbhd"] = name
    osm_results.append(features)

osm_destinations = pd.concat(osm_results, ignore_index=True)
print(f"Total raw OSM features pulled: {len(osm_destinations)}")

Total raw OSM features pulled: 2758


In [33]:
def assign_category(row):
    for key in ["amenity", "shop"]:
        val = row.get(key)
        if val is not None and (key, val) in category_map:
            return category_map[(key, val)]
    if row.get("shop") is not None:
        return ("Shop", 1, 2)  # catch-all for unlisted shop values
    return (None, None, None)

osm_destinations[["category", "weight_min", "weight_max"]] = osm_destinations.apply(
    lambda row: pd.Series(assign_category(row)), axis=1
)

osm_destinations = osm_destinations[osm_destinations["category"].notna()]
print(osm_destinations["category"].value_counts())

category
Shop          1811
Restaurant     357
Shopping       262
Stopby         227
Grocery         53
Kids            48
Name: count, dtype: int64


In [34]:
print(osm_destinations[osm_destinations["category"] == "Shop"]["shop"].value_counts())

shop
deli             49
laundry          48
alcohol          42
variety_store    27
mobile_phone     25
                 ..
bed               1
newsagent         1
charity           1
boutique          1
street_vendor     1
Name: count, Length: 65, dtype: int64


In [35]:
osm_destinations["geometry"] = osm_destinations.geometry.apply(
    lambda geom: geom.centroid if geom.geom_type != "Point" else geom
)

In [36]:
import os
os.makedirs("outputs", exist_ok=True)

for category in osm_destinations["category"].unique():
    subset = osm_destinations[osm_destinations["category"] == category]
    filename = f"outputs/osm_{category.lower()}.geojson"
    subset.to_file(filename, driver="GeoJSON")
    print(f"{category}: {len(subset)} features -> {filename}")

Kids: 48 features -> outputs/osm_kids.geojson
Shop: 1811 features -> outputs/osm_shop.geojson
Restaurant: 357 features -> outputs/osm_restaurant.geojson
Shopping: 262 features -> outputs/osm_shopping.geojson
Stopby: 227 features -> outputs/osm_stopby.geojson
Grocery: 53 features -> outputs/osm_grocery.geojson


In [37]:
import os
os.makedirs("outputs", exist_ok=True)
osm_destinations.to_file("outputs/osm_destinations.geojson", driver="GeoJSON")

Final export to shapefile for Rhino Upload

In [38]:
subway_export = subwaystops[["geometry", "CBD"]]  # add other short-named columns as needed
subway_export = subway_export.to_crs("EPSG:4326")
subway_export.to_file("outputs/subway_stops.shp")

In [40]:
export_cols = {
    "Route Type": "rte_type"
}

bstops_export = bstops.rename(columns=export_cols)

if "neighborhood" in bstops_export.columns:
    bstops_export["nbhd"] = bstops_export["neighborhood"]
elif "nbhd" not in bstops_export.columns:
    bstops_export["nbhd"] = ""

bstops_export = bstops_export[["geometry", "rte_type", "nbhd"]]
bstops_export = bstops_export.to_crs("EPSG:4326")
bstops_export.to_file("outputs/bus_stops.shp")

In [41]:
export_cols = {
    "neighborhood": "nbhd"
}

osm_export = osm_destinations.rename(columns=export_cols)
if "neighborhood" in osm_destinations.columns:
    osm_export["nbhd"] = osm_destinations["neighborhood"]
elif "nbhd" not in osm_export.columns:
    osm_export["nbhd"] = ""

osm_export = osm_export[["geometry", "category", "weight_min", "weight_max", "nbhd"]]
osm_export = osm_export.to_crs("EPSG:4326")
osm_export.to_file("outputs/osm_destinations.shp")